<a href="https://colab.research.google.com/github/Tommy-Burns/Machine_Learning_Project/blob/main/EuroCropMilestone1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Crop Type Classification from Sentinel-2 Time Series**

EuroCropsML combines EuroCrops (v9) parcel data with 2021 Sentinel-2 reflectance data from Latvia, Portugal, and Estonia for few-shot crop classification benchmarking


## Installations

In [ ]:
!pip install -q huggingface_hub

## Imports

In [ ]:
from huggingface_hub import hf_hub_download
import zipfile, io
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import geopandas as gpd
from tqdm.auto import tqdm

# plot styling
sns.set_theme(style="whitegrid", rc={
    "axes.edgecolor": "#c3c2b7", "grid.color": "#e1e0d9",
    "axes.labelcolor": "#52514e", "xtick.color": "#52514e", "ytick.color": "#52514e",
    "axes.titlesize": 12, "axes.titleweight": "bold",
})
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
MUTED = "#c3c2b7"
INK = "#52514e"

## Helper Functions


In [ ]:
FILES = {
    "preprocess": "preprocess/portugal.zip",
    "labels": "raw_data/labels/Portugal_labels.parquet",
    "classes": "cropclasses.csv",
}

def fetch_data(file_name, data_dir="data"):
    path = Path(data_dir) / FILES[file_name]

    if not path.exists():
      path = Path(
          hf_hub_download(
              repo_id="tbbotchwey/eurocropsml",
              filename=FILES[file_name],
              repo_type="dataset",
              local_dir=data_dir,
          )
        )
    if file_name == "preprocess":
      return zipfile.ZipFile(path)
    if file_name == "geometry":
      return gpd.read_file(path)
    if file_name == "classes":
      return pd.read_csv(path)
    return pd.read_parquet(path)


# GLOBAL FILENAMES
preprocess = fetch_data("preprocess")
labels  = fetch_data("labels")
classes = fetch_data("classes")

## Extract Preprocess Data

Flatten every `.npz` in the preprocess zip into one table with one row per parcel and observation date.

Each file is named `{NUTS3}_{parcel_id}_{HCAT code}.npz` and holds:
- `data`: dates × 13 Sentinel-2 L1C bands
- `dates`: observation dates
- `center`: parcel midpoint (lon, lat)

An observation date is dropped when B02/B03/B04 are all 0 or B05/B06/B07 are all 0.
Parcels left without any valid date are dropped.

In [ ]:
BANDS = ["B01", "B02", "B03", "B04", "B05", "B06", "B07",
         "B08", "B8A", "B09", "B10", "B11", "B12"]

def all_zero(data, bands):
    idx = [BANDS.index(b) for b in bands]
    return np.all(data[:, idx] == 0, axis=1)

all_npz_files = sorted(n for n in preprocess.namelist() if n.endswith(".npz"))

npz_files, parcels, dates, values = [], [], [], []
for name in tqdm(all_npz_files, desc="Reading NPZ files", mininterval=15):
    file_name = Path(name).name
    nuts3, parcel_id, hcat_code = file_name[:-4].split("_")

    with np.load(io.BytesIO(preprocess.read(name))) as npz:
        data = npz["data"].astype(np.int16)
        parcel_dates = npz["dates"]

        # invalid if B02/B03/B04 are all 0, OR B05/B06/B07 are all 0
        invalid = all_zero(data, ("B02", "B03", "B04")) | all_zero(data, ("B05", "B06", "B07"))
        valid_dates = ~invalid
        if not valid_dates.any():
            continue

        # store the valid values
        lon, lat = npz["center"]
        npz_files.append(file_name)
        parcels.append((int(parcel_id), file_name, lon, lat, int(hcat_code), nuts3, int(valid_dates.sum())))
        dates.append(parcel_dates[valid_dates])
        values.append(data[valid_dates])

parcels = pd.DataFrame(parcels, columns=[
    "parcel_id", "file_name", "midpoint_lon", "midpoint_lat",
    "hcat_crop_code", "nuts3", "n_observations"
    ])
print(f"Dropped {len(all_npz_files) - len(npz_files):,} parcels with no valid dates")
print(f"{len(parcels):,} parcels, {parcels['n_observations'].sum():,} observations")

Attach crop names from `labels` and the HCAT level-3 class from `classes`.

In [ ]:
crop_labels = labels.drop_duplicates("parcel_id").set_index("parcel_id")
level3 = classes.set_index("HCAT3_level6_code")

parcels["hcat_crop_name"] = parcels["parcel_id"].map(crop_labels["EC_hcat_n"])
parcels["hcat_level3_code"] = parcels["hcat_crop_code"].map(level3["HCAT3_level3_code"])
parcels["hcat_level3_name"] = parcels["hcat_crop_code"].map(level3["HCAT3_level3_name"])

# check that the HCAT code in the file name agrees with the labels file
label_code = parcels["parcel_id"].map(crop_labels["EC_hcat_c"]).astype("int64")
print("code mismatches:", (label_code != parcels["hcat_crop_code"]).sum())
print("missing values:", int(parcels.isna().sum().sum()))

Expand to one row per observation date and add the band values.

In [ ]:
df = parcels.loc[parcels.index.repeat(parcels["n_observations"])].reset_index(drop=True)
df["date"] = np.concatenate(dates).astype("datetime64[D]")
df["day_of_year"] = df["date"].dt.dayofyear
df = pd.concat([df, pd.DataFrame(np.concatenate(values), columns=BANDS)], axis=1)

del values  # band values are now df and the list is no longer needed

df = df[["parcel_id", "file_name", "date", "day_of_year", "midpoint_lon", "midpoint_lat",
         "hcat_crop_code", "hcat_crop_name", "hcat_level3_code", "hcat_level3_name",
         "nuts3", "n_observations", *BANDS]]
print(df.shape)
df.head()

In [ ]:
out_path = Path("output/portugal_preprocess_long.parquet")
out_path.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(out_path, index=False)
print(f"Saved {out_path}")

## Select Parcels by Temporal Coverage

The valid dates are grouped into bins of a fixed number of days, counted from the first date (2021-01-01).
For each bin, count the *known* parcels (everything except `not_known_and_other`) that have at least one date in it.

- A parcel is **kept** only if it has at least one date in every required bin (this applies to `not_known_and_other` parcels too).

The selection uses 15-day bins and an 85% threshold, matching the 15-day resampling below.
The appendix shows every bin size and threshold (`BIN_SIZES` and `THRESHOLDS` variables in the next cell).

In [ ]:
BIN_SIZES = [1, 5, 10, 15]
THRESHOLDS = [0.80, 0.85, 0.90, 0.95]
THRESHOLD_COLORS = ["#eb6834", "#1baf7a", "#4a3aa7", "#e87ba4"]  # one per threshold line

UNKNOWN_CODE = "3399000000"   # not_known

def hcat3(name: str) -> str:
    return name.rsplit("/", 1)[-1].removesuffix(".npz").rsplit("_", 1)[-1]

def summarize_dates(files, dates_per_file, bin_sizes = BIN_SIZES, thrs = THRESHOLDS, label="Portugal", show=True, unknown_always_kept=False):
    n_files = len(files)

    known = np.array([hcat3(f) != UNKNOWN_CODE for f in files])
    n_known = int(known.sum())

    lengths = np.fromiter(map(len, dates_per_file), dtype=int, count=n_files)
    all_dates = np.concatenate(dates_per_file)
    file_idx = np.repeat(np.arange(n_files), lengths)

    origin = all_dates.min()
    offsets = (all_dates - origin).astype(int)
    n_days = offsets.max() + 1

    results = {}
    kept_files = {}
    required_bins = {}
    for size in bin_sizes:
        n_bins = -(-n_days // size)
        presence = np.zeros((n_files, n_bins), dtype=bool)
        presence[file_idx, offsets // size] = True
        known_counts = presence[known].sum(axis=0)

        fig, ax = plt.subplots(figsize=(14, 4))
        ax.bar(np.arange(n_bins), known_counts, color=SERIES[0])
        for i, t in enumerate(thrs):
            ax.axhline(t * n_known, ls="--", color=THRESHOLD_COLORS[i % len(THRESHOLD_COLORS)],
                      label=f"{int(t*100)}% ({t*n_known:.1f} known files)")
        ax.set(title=f"{label}: files with ≥1 date per {size}-day group",
               xlabel=f"{size}-day group index (from {origin})",
               ylabel="Number of known files")
        ax.legend()
        fig.tight_layout()
        plt.show() if show else plt.close(fig)

        for t in thrs:
            kept_bins = known_counts >= t * n_known
            kept_mask = presence[:, kept_bins].all(axis=1)   # applies to all files
            if unknown_always_kept:
                kept_mask |= ~known
            results[(size, t)] = int(kept_mask.sum())
            kept_files[(size, t)] = [f for f, k in zip(files, kept_mask) if k]
            required_bins[(size, t)] = np.flatnonzero(kept_bins)
            n_kept_known = int((kept_mask & known).sum())
            print(f"[{label}] {size}-day, ≥{int(t*100)}% of known: "
                  f"{kept_bins.sum()} bins kept, "
                  f"{n_kept_known}/{n_known} known + "
                  f"{results[(size, t)] - n_kept_known}/{n_files - n_known} not_known files kept")

    return {"n_files": n_files, "n_known": n_known, "files": files,
            "all_dates": all_dates, "results": results, "kept_files": kept_files,
            "required_bins": required_bins}

In [ ]:
KEPT_BIN_SIZE, KEPT_THRESHOLD = 15, 0.85

coverage = summarize_dates(npz_files, dates, [KEPT_BIN_SIZE], [KEPT_THRESHOLD], show=False)
kept = coverage["kept_files"][(KEPT_BIN_SIZE, KEPT_THRESHOLD)]
print(f"{len(kept):,} parcels kept")

## Filter to Kept Parcels

In [ ]:
df_kept = df[df["file_name"].isin(kept)].reset_index(drop=True)

print("kept files not found in df:", len(set(kept) - set(parcels["file_name"])))
print(f"{df_kept['parcel_id'].nunique():,} of {len(parcels):,} parcels, {len(df_kept):,} rows")

del df  # the full table is saved in output/portugal_preprocess_long.parquet
df_kept.head()

In [ ]:
kept_path = Path("output/portugal_preprocess_kept.parquet")
df_kept.to_parquet(kept_path, index=False)
print(f"Saved {kept_path}")

### Clean

- drop parcels labelled `not_known_and_other`
- drop the remaining observations with a 0 in any of the 10 m / 20 m bands
- drop B01, B09 and B10: 60 m atmospheric bands (aerosol, water vapour, cirrus)

In [ ]:
BANDS_ = ["B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B11", "B12"]

keep_rows = (df_kept["hcat_crop_name"] != "not_known_and_other") & (df_kept[BANDS_] != 0).all(axis=1)
df_clean = df_kept.loc[keep_rows, df_kept.columns.drop(["B01", "B09", "B10"])].reset_index(drop=True)

print(f"{df_clean['parcel_id'].nunique():,} parcels, {len(df_clean):,} rows "
      f"({len(df_kept) - len(df_clean):,} rows removed)")
print("0s left in band columns:", int((df_clean[BANDS_] == 0).sum().sum()))

## Resample to 15-day Periods

One row per parcel and 15-day period that has data (24 periods. days 361-365 are folded into the last one).
Band values are the **median** of the observations in each period:
- clouds are already masked out in the dataset, so mean and median give very similar results
- with 1-2 observations (the most common case) the median equals the mean
- with 3+ observations the median ignores a single outlier the cloud mask missed (e.g. cloud shadow or haze), which the mean would average in

In [ ]:
PERIOD_DAYS = KEPT_BIN_SIZE      # same bins as the parcel selection
N_PERIODS = 365 // PERIOD_DAYS   # days 361-365 are folded into the last period

df_clean["period"] = np.minimum((df_clean["day_of_year"] - 1) // PERIOD_DAYS, N_PERIODS - 1)

grouped = df_clean.groupby(["parcel_id", "period"])
df_15d = grouped[BANDS_].median()
df_15d["n_obs_in_period"] = grouped.size()

# keep only periods with at least one observation (parcels can have fewer than N_PERIODS rows)
df_15d = df_15d.reset_index()
df_15d["n_obs_in_period"] = df_15d["n_obs_in_period"].astype("int16")
df_15d["period_start_doy"] = df_15d["period"] * PERIOD_DAYS + 1
df_15d["period_start"] = pd.Timestamp("2021-01-01") + pd.to_timedelta(df_15d["period_start_doy"] - 1, unit="D")

parcel_info = df_clean.drop_duplicates("parcel_id").set_index("parcel_id")[
    ["file_name", "midpoint_lon", "midpoint_lat", "hcat_crop_code", "hcat_crop_name",
     "hcat_level3_code", "hcat_level3_name", "nuts3", "n_observations"]]
df_15d = df_15d.join(parcel_info, on="parcel_id")

df_15d = df_15d[["parcel_id", "file_name", "period", "period_start", "period_start_doy",
                 "midpoint_lon", "midpoint_lat", "hcat_crop_code", "hcat_crop_name",
                 "hcat_level3_code", "hcat_level3_name", "nuts3", "n_observations",
                 "n_obs_in_period", *BANDS_]]

periods_per_parcel = df_15d.groupby("parcel_id").size()
print(df_15d.shape)
print(f"{df_15d['parcel_id'].nunique():,} parcels, {periods_per_parcel.min()}-{periods_per_parcel.max()} periods each")
print(f"parcels with all {N_PERIODS} periods: {(periods_per_parcel == N_PERIODS).sum():,}")

In [ ]:
path_15d = Path("output/portugal_kept_15day_median.parquet")
df_15d.to_parquet(path_15d, index=False)
print(f"Saved {path_15d}")

## Map it such that every parcel id appears once

In [ ]:
# keep only the periods every parcel has data for
required_periods = sorted(set(np.minimum(coverage["required_bins"][(KEPT_BIN_SIZE, KEPT_THRESHOLD)], N_PERIODS - 1)))
df_req = df_15d[df_15d["period"].isin(required_periods)]

# one column per band and period, ordered period by period: B02_p00...B12_p00, B02_p03...B12_p23
wide = df_req.pivot(index="parcel_id", columns="period", values=BANDS_)
band_cols = [(band, period) for period in required_periods for band in BANDS_]
wide = wide[band_cols]
wide.columns = [f"{band}_p{period:02d}" for band, period in band_cols]

parcel_cols = ["file_name", "midpoint_lon", "midpoint_lat", "hcat_crop_code", "hcat_crop_name",
               "hcat_level3_code", "hcat_level3_name", "nuts3", "n_observations"]
df_wide = (df_15d.drop_duplicates("parcel_id").set_index("parcel_id")[parcel_cols]
           .join(wide).reset_index())

print(df_wide.shape)
print(f"{len(required_periods)} periods x {len(BANDS_)} bands = {wide.shape[1]} band columns")
print("missing values:", int(df_wide.isna().sum().sum()))

# df_wide.head()

In [ ]:
path_wide = Path("output/portugal_kept_15day_wide.parquet")
df_wide.to_parquet(path_wide, index=False)
print(f"Saved {path_wide}")

## Exploratory Data Analysis

Descriptive statistics and four plots of the final 15-day dataset (`df_15d`).

In [ ]:
parcel_level = df_15d.drop_duplicates("parcel_id")
periods_per_parcel = df_15d.groupby("parcel_id").size()
obs = df_15d["n_obs_in_period"]

pd.Series({
    "parcels": f"{len(parcel_level):,}",
    "rows (parcel x 15-day period)": f"{len(df_15d):,}",
    "observations before resampling": f"{obs.sum():,}",
    "crop types (HCAT level 6)": parcel_level["hcat_crop_name"].nunique(),
    "crop groups (HCAT level 3)": parcel_level["hcat_level3_name"].nunique(),
    "NUTS3 regions": parcel_level["nuts3"].nunique(),
    "periods per parcel (min / median / max)":
        f"{periods_per_parcel.min()} / {periods_per_parcel.median():.0f} / {periods_per_parcel.max()}",
    "observations per period (min / median / max)": f"{obs.min()} / {obs.median():.0f} / {obs.max()}",
}).to_frame("value")

Band values

In [ ]:
df_15d[BANDS_].describe().T

Class balance (parcels per crop type):

In [ ]:
crop_counts = parcel_level["hcat_crop_name"].value_counts()
print(f"{(crop_counts < 100).sum()} of {len(crop_counts)} crop types have fewer than 100 parcels")

(crop_counts.to_frame("parcels")
 .assign(share_pct=lambda t: (t["parcels"] / t["parcels"].sum() * 100).round(1))
 .head(10))

**Observation**  
The classes are very imbalanced: pasture (26.4%) and olive plantations (23.0%) make up half of the parcels, and 45 of the 71 crop types have fewer than 100 parcels

### 1. Parcels with data in each 15-day period

Number of final parcels with at least one observation per 15-day period.
Blue bars are the periods the parcel selection required; the dashed lines mark 80-95% of the final parcels.

In [ ]:
required = set(np.minimum(coverage["required_bins"][(KEPT_BIN_SIZE, KEPT_THRESHOLD)], N_PERIODS - 1))
n_parcels = df_15d["parcel_id"].nunique()

per_period = (df_15d.groupby("period")["parcel_id"].nunique()
              .reindex(range(N_PERIODS), fill_value=0).rename("parcels").reset_index())
per_period["period_start"] = (pd.Timestamp("2021-01-01")
                              + pd.to_timedelta(per_period["period"] * PERIOD_DAYS, unit="D")).dt.strftime("%d %b")
REQUIRED, NOT_REQUIRED = "required by the selection", "not required"
per_period["selection"] = np.where(per_period["period"].isin(required), REQUIRED, NOT_REQUIRED)

fig, ax = plt.subplots(figsize=(13, 4.5))
sns.barplot(per_period, x="period_start", y="parcels", hue="selection", hue_order=[REQUIRED, NOT_REQUIRED],
            palette={REQUIRED: SERIES[0], NOT_REQUIRED: MUTED}, dodge=False, ax=ax)
for t, color in zip(THRESHOLDS, THRESHOLD_COLORS):
    ax.axhline(t * n_parcels, ls="--", lw=1, color=color, label=f"{t:.0%} of parcels")
ax.set_ylim(0, n_parcels * 1.05)
ax.set(title=f"Parcels with ≥1 observation per {PERIOD_DAYS}-day period (final dataset, n = {n_parcels:,})",
       xlabel="Period start", ylabel="Parcels")
ax.tick_params(axis="x", rotation=45)
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=9)
fig.tight_layout()
plt.show()

**Observations**
- All 48,269 parcels have data in every period the selection required (blue bars).
- The four periods that were not required have lower coverage: 16 Jan (55%), 31 Jan (85%), 16 Apr (84%) and 29 Aug (92%). These are the gaps where some parcels have no row.

### 2. Crop type distribution

Top crop types by number of parcels, colored by HCAT level-3 group (adapted from `plot_top_classes_color_l3`).

In [ ]:
TOP_N = 20

crop_l3 = (parcel_level.groupby(["hcat_crop_name", "hcat_level3_name"]).size()
           .rename("parcels").reset_index().sort_values("parcels", ascending=False))
level3_order = crop_l3.groupby("hcat_level3_name")["parcels"].sum().sort_values(ascending=False).index.tolist()
level3_color = dict(zip(level3_order, SERIES))   # fixed color per level-3 group
top = crop_l3.head(TOP_N)

fig, ax = plt.subplots(figsize=(10, 7))
sns.barplot(top, y="hcat_crop_name", x="parcels", hue="hcat_level3_name",
            hue_order=[g for g in level3_order if g in set(top["hcat_level3_name"])],
            palette=level3_color, dodge=False, ax=ax)
ax.set_xscale("log")
ax.set_xlim(right=top["parcels"].max() * 3)
for container in ax.containers:
    ax.bar_label(container, fmt="{:,.0f}", padding=3, fontsize=8, color=INK)
ax.set(title=f"Top {TOP_N} of {len(crop_l3)} crop types", xlabel="Parcels (log scale)", ylabel="")
ax.legend(title="HCAT level 3", loc="lower right")
fig.tight_layout()
plt.show()

**Observations**
- The 20 most common crop types come from 4 HCAT level-3 groups. Most are permanent crops (olives, vineyards, chestnuts, fruit trees) or arable crops. pasture is a single crop type but the largest one.
- Below the top 10, crop types quickly drop to a few hundred parcels

### 3. NDVI profiles of the most common crop types

NDVI = (B08 − B04) / (B08 + B04), computed here only for the plot.

In [ ]:
TOP_CROPS = 6
top_crops = crop_counts.head(TOP_CROPS).index.tolist()

sub = df_15d[df_15d["hcat_crop_name"].isin(top_crops)]
ndvi = sub[["hcat_crop_name", "period_start"]].assign(ndvi=(sub["B08"] - sub["B04"]) / (sub["B08"] + sub["B04"]))
profile = ndvi.groupby(["hcat_crop_name", "period_start"])["ndvi"].quantile([0.25, 0.5, 0.75]).unstack()
profile.columns = ["q25", "median", "q75"]
profile = profile.reset_index()

crop_color = dict(zip(top_crops, SERIES))
fig, ax = plt.subplots(figsize=(12, 5))
for crop in top_crops:
    p = profile[profile["hcat_crop_name"] == crop]
    ax.fill_between(p["period_start"], p["q25"], p["q75"], color=crop_color[crop], alpha=0.12, lw=0)
sns.lineplot(profile, x="period_start", y="median", hue="hcat_crop_name", hue_order=top_crops,
             palette=crop_color, lw=2, ax=ax)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
ax.set(title="Median NDVI per 15-day period (band: interquartile range across parcels)",
       xlabel="Period start (2021)", ylabel="NDVI")
ax.legend(title="Crop type", bbox_to_anchor=(1.01, 1), loc="upper left")
fig.tight_layout()
plt.show()

**Observations**
- Maize has a distinct summer peak (median NDVI apprx 0.66 in July-August)
- Sweet chestnuts stay low in winter and green up from May
- Pasture, temporary grass, olives and vineyards overlap a lot
- The dips on 16 Jan and late August affect all crops at once. both are periods with lower coverage, so they may reflect acquisition conditions rather than crop growth.

## Appendix: Temporal Coverage by Bin Size and Threshold

For each bin size (1, 5, 10 and 15 days): the number of known parcels (excluding `not_known_and_other`) with at least one valid date per bin.
The dashed lines are the 80%, 85%, 90% and 95% thresholds. The printed lines give, per threshold, how many bins are required and how many parcels would be kept.

This uses all parcels with at least one valid date, before the parcel selection.

In [ ]:
appendix = summarize_dates(npz_files, dates, BIN_SIZES, THRESHOLDS, label="Portugal", show=True)

Parcels kept (known and `not_known_and_other`) for each bin size and threshold:

In [ ]:
(pd.Series(appendix["results"]).rename_axis(["bin size (days)", "threshold"]).unstack()
 .rename(columns=lambda t: f"≥{t:.0%}"))

#**Data Introduction**



### **1.1 How and Why data was collected**
The data comes from **EuroCropsML**, developed by TU Munich and Dida within the PretrainAppEO project.
It uses data from two sources:
- Labels from Eurocrops (V9), which is an open collection of agricultural reference data from EU countries
- Satellite observations from Sentinel-2 L1C top of atmosphere reflectance for 2021, aggregated per parcel as the median pixel value inside the parcel geometry

**EuroCropsML** was built as a benchmark for few-shot crop type classification in 3 countries, but for our project we use it as a standard supervised classification dataset on one country: Portugal.

The data is as follows:
- **Raw data:** Contains parquet for each country, labels and geometries (all observations)
- **Preprocessed data:**
    - Ready to use .npz files, one per data point with metdata (parcel centre, dates and data)
    - Clouds have been removed
    - file names follow the pattern "[NUTS3-region]\_[parcelID]\_[HCAT-code].npz" ( e.g. "PT16E_1196198_3399000000.npz")
- **Split data:** json files that divide the preprocessed files into a pre-training (train and validation set) set and a fine tuning (train,validation, test) set for 2 scenarios:
    - latvia_portugal_vs_estonia: pre-training on Latvia and Portugal, fine-tuning on Estonia
    - latvia_vs_estonia: pre-training on Latvia and fine-tuning on Estonia

Since we were not interested in any of the scenarios provided by the split data, we decided to work with the preprocessed data.

**Geographic Coverage:** Latitude 37.3 to 42.1 and longitude -9.45 to -6.47, which covers most of Portugal's mainland, but not the islands, as shown below.

<img src="https://i.imgur.com/2rwa3O5.png" width="400">

### **1.2 Application Domain**
Agricultural monitoring with Earth Observation

### **1.2 Learning Task**
Supervised multi-class Crop type classification: for every parcel based on its annual Sentinel-2 time series, we predict its crop type

In [ ]:
_ = summarize_dates2(npz_files, dates, [15], [0.85])

### **1.3 Features**
Each parcel is stored as one .npz file with three arrays
- Data: A list of various observation dates * 13 Sentinel-2 Bands, consisting of non-negative refectance values
- Dates: Acquisition date of each row
- Center: Parcel centroid

After carefully analyzing the data, we decided to aggregate the dates in 15-day intervals (as shown in the figure above), to ensure that all the samples would have the same features. Furthermore, we also dropped 3 Bands (B01, B09 and B10), since a large number of parcels were missing data for these, and they are not paramount for vegetation classification. Thus, the final list of features should contain a column for each of the 10 remaining band for each of our 20 15-day intervals.

### **1.4 Output Variable**
The HCAT crop type of a parcel, a nominal categorical variable (multi-class, no ranking, classification), which means our project should be able to give one label output from a fixed list of crop type.

In [ ]:
plot_top_classes_color_l3(kept, "Top 30 crop types (kept parcels, 15-day bins, ≥85%)", 30)

# **Data Description**

Potential Pre-processing
- We choose to use stratified shuffled sampling with a seed (specified random state) to ensure reproducibility. Since we have no data in the islands, this should be enough to get diversified samples.
- As mentioned before, we have also aggregated the data by 15-day periods, which meant we had to calculate the average of observations that fell inside the same period.
- Since we are only using Sentinel-2 band values, there should be no need to scale the data, regardless of the model used.


# **Evaluation Protocol**

Our goal is to build a model that is able to predict the crop type of parcels it has never seen, without information leaking from test to training.

### **4.1 Data Splitting**
- Test set: 70%
- Test 10%
- Validation 20%

We plan to use the cross validation, we are yet to decide on the folds but we will start with the default 5, where we have 4 training folds and one hold out.


### **4.2 Hyperparameter Tuning**
For each algorithm, we will identify its main hyperparameters, e.g. number of tress for random forest, and tune those with the validation set.

### **4.3 Planned Algorithms**
- K-Neighbors
- Random Forest
- SVM


### **4.4 Evaluation Metrics**
Since the output is a categorical crop label instead of a numeric value, we will measure how often the model is right, and for which crops, instead of how far off its predictions are. We plan to use:

- **Overall accuracy:** the share of all parcels classified correctly;
- **Per-class accuracy:** for each crop type, the share of all parcels that are handled correctly in a one-vs-rest sense.
- **Per-class precision:** for each crop type, how many of the predicted parcels actually belong to it;
- **Per-class recall:** for each crop type, how many of its parcels the model actually finds;
- **Macro F1:** the unweighted average of per-class F1 scores (which is the harmonic mean of precision and recall), thus treating every crop type equally.

Because the classes are highly imbalanced, overall accuracy can look good simply by predicting the most common crops, and per-class accuracy is inflated for rare crops, since most parcels are correctly "not" that class. For this reason, recall, precision and macro F1 will carry more weight in our conclusions than the two accuracy measures.

One caveat: the crop types with very few parcels (possibly in a range of 1 to 15 in the dataset), will not be taken "seriously" when evaluating the models. With so few samples, their metrics are extremely unstable, and (in the extreme cases) they may not even appear in every fold of the cross validation.